# Serverless PROJ grid-registration validation

Proves the **light-tier grid-dir propagation to real Serverless executors** end to end.

The source CRS references a synthetic NTv2 grid by filename (`+nadgrids=synthetic.gsb`), which
applies a known **+30 arc-second latitude** shift. PROJ can only build the transform if pyproj on
the *executor* locates `synthetic.gsb` on its search path. The only way that happens on a worker
is if `register_proj_grids` embedded the grid dir in the pickled `gbx_st_transformcrs` UDF closure
and `configure_gdal_env(extra_proj_dirs=...)` pushed it into the worker's pyproj at task time.

We fan a point across 8 partitions with `repartition(8, "id")` (the only supported Serverless
light-parallelism lever) so multiple executor tasks each independently build a transformer and must
each find the grid. Getting the exact shift on **every** partition is the proof the closure carried
the dir to the executors.

In [ ]:
import json

GRID_DIR = "/Volumes/geospatial_docs/geobrix/sample-data/proj-grids"
GRID_FILE = GRID_DIR + "/synthetic.gsb"

SRC_CRS = "+proj=longlat +ellps=GRS80 +nadgrids=synthetic.gsb +no_defs"
TGT_CRS = "EPSG:4326"
PT_WKT = "POINT (0 51.5)"
SHIFT_SEC = 30.0
EXPECT_LON = 0.0
EXPECT_LAT = 51.5 + SHIFT_SEC / 3600.0  # 51.508333...
N_PART = 8

results = {"grid_file": GRID_FILE, "expect_lat": EXPECT_LAT, "stages": {}}

import os

assert os.path.isfile(GRID_FILE), f"grid fixture not staged/visible on driver: {GRID_FILE}"
print("driver sees grid fixture:", GRID_FILE, os.path.getsize(GRID_FILE), "bytes")

In [ ]:
# Register the light VectorX SQL functions, then register the grid dir.
# register_proj_grids re-registers gbx_st_transformcrs with GRID_DIR embedded in the closure.
from databricks.labs.gbx.pyvx import functions as pyvx_functions
from databricks.labs.gbx import crs_grids
from databricks.labs.gbx.core import proj_grids

proj_grids.set_registered_dirs([], replace=True)  # clean slate
pyvx_functions.register(spark, only=["gbx_st_transformcrs"])
registered = crs_grids.register_proj_grids(spark, GRID_DIR)
print("registered dirs:", registered)
results["registered"] = registered
assert GRID_DIR in registered

In [ ]:
# Fan the grid-required transform across executors and collect WKB results.
from pyspark.sql import functions as F
from shapely import wkb as _wkb

df = (
    spark.range(N_PART)
    .withColumn("id", F.col("id").cast("int"))
    .withColumn("geom", F.lit(PT_WKT))
    .repartition(N_PART, "id")  # force distribution across executor tasks
    .selectExpr(
        "id",
        "spark_partition_id() AS pid",
        f"gbx_st_transformcrs(geom, '{TGT_CRS}', '{SRC_CRS}') AS out_wkb",
    )
)
rows = df.collect()
print(f"collected {len(rows)} rows across {len(set(r['pid'] for r in rows))} partitions")

lats = []
for r in rows:
    assert r["out_wkb"] is not None, f"transform returned NULL on partition {r['pid']} — grid not found on that executor"
    g = _wkb.loads(bytes(r["out_wkb"]))
    lats.append((r["pid"], g.x, g.y))
results["stages"]["treated"] = [{"pid": p, "lon": x, "lat": y} for (p, x, y) in lats]
for p, x, y in lats:
    print(f"  pid={p}  lon={x:.8f}  lat={y:.8f}")

In [ ]:
# Assert the exact +30 arc-second shift landed on EVERY partition.
TOL = 1e-6
bad = [
    (p, x, y)
    for (p, x, y) in lats
    if abs(x - EXPECT_LON) > TOL or abs(y - EXPECT_LAT) > TOL
]
results["n_partitions"] = len(set(p for p, _, _ in lats))
results["n_rows"] = len(lats)
results["bad"] = bad
results["pass"] = not bad
print(json.dumps(results, indent=2))
assert not bad, (
    f"{len(bad)} partition(s) did NOT get the registered-grid shift (expected lat {EXPECT_LAT}); "
    f"the grid dir did not propagate to those executors: {bad}"
)
print("\nPASS: registered PROJ grid was consulted on all", results["n_partitions"], "executor partitions.")
dbutils.notebook.exit(json.dumps(results))  # surface the auditable result to jobs.get_run_output